# Installing libraries

In [ ]:
!pip install -q transformers accelerate

# Mouting google drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Importing the model from google drive

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/content/drive/MyDrive/book-lora-merged"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

#Ask question

In [5]:
model.eval()
def ask(question, context=None, max_new_tokens=300):
    # Use the chat template if the tokenizer has one, otherwise a plain prompt
    user_msg = question if context is None else f"Context:\n{context}\n\nQuestion: {question}"

    if tokenizer.chat_template:
        messages = [{"role": "user", "content": user_msg}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
    else:
        prompt = f"{user_msg}\n\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic answers are best for QA
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [7]:
print(ask("What is mahabharata"))

The Mahabharata is one of the two most important epic Hindu texts in Indian literature. It is traditionally believed to have been composed around 100 BCE and is considered the oldest extant work of classical Sanskrit literature.

The Mahabharata tells the story of the Kaurava warriors (also known as the Pandavas) who fought against the Duryodhana and his army for control of the kingdom of Hastinapura. The text also includes stories about the Bhima, the king of the Pandavas, and the Kurukshetra war between the Pandavas and their enemies.

The Mahabharata has been widely studied and translated into various languages over the centuries, making it an important source of knowledge for scholars and students alike. It is often referred to as "the eternal epic" due to its enduring popularity and cultural significance.


#Chat UI

In [8]:
import gradio as gr
gr.ChatInterface(lambda message, history: ask(message), title="Ask the Book").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7c30e67d795899294e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
